# 🧠 NeuroFlow - AI-Powered Study Planner

**Achieve 90%+ scores without burnout**

NeuroFlow creates optimized daily study schedules based on your energy cycles, academic goals, subject difficulty, and exam timelines.

---

## 🚀 Quick Start

1. Run the setup cell below
2. Enter your Google API Key
3. Fill in your study details
4. Click "Generate My Study Plan"

---

## 📦 Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q gradio==4.12.0 google-generativeai==0.3.2 reportlab==4.0.7

print("✅ Dependencies installed successfully!")

## 🔑 Step 2: Set Your Google API Key

Get your free API key from: https://makersuite.google.com/app/apikey

In [ ]:
import os

# 🔴 Replace with your actual API key
GOOGLE_API_KEY = "YOUR_API_KEY_HERE"

# Set environment variable
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

if GOOGLE_API_KEY != "YOUR_API_KEY_HERE":
    print("✅ API Key configured!")
else:
    print("⚠️ Please enter your API key above")

## 📁 Step 3: Create Application Files

In [ ]:
# Create config.py
config_content = '''
"""NeuroFlow Configuration"""

import os

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
GEMINI_MODEL = "gemini-1.5-flash"
GEMINI_TEMPERATURE = 0.3

APP_NAME = "NeuroFlow"
APP_TAGLINE = "AI-Powered Adaptive Study Planner"
APP_VERSION = "1.0.0"

ENERGY_PATTERNS = {
    "morning_high": "Morning High (Early Bird)",
    "afternoon_high": "Afternoon High",
    "night_high": "Night Owl",
    "bimodal": "Bimodal (Morning + Evening)",
    "consistent": "Consistent Energy"
}

BREAK_STRUCTURES = {
    "Pomodoro (25-5)": (25, 5),
    "Deep Focus (50-10)": (50, 10),
    "Ultra Deep (90-20)": (90, 20),
    "Micro Learning (15-3)": (15, 3)
}
'''

with open('config.py', 'w') as f:
    f.write(config_content)

print("✅ config.py created")

In [ ]:
# Create simplified AI engine
ai_engine_content = '''
"""AI Engine for NeuroFlow"""

import json
import re
import google.generativeai as genai
from config import GOOGLE_API_KEY, GEMINI_MODEL, GEMINI_TEMPERATURE

class GeminiAIEngine:
    def __init__(self):
        self.api_key = GOOGLE_API_KEY
        self.model = None
        if self.api_key:
            genai.configure(api_key=self.api_key)
            self.model = genai.GenerativeModel(GEMINI_MODEL)
    
    def is_ready(self):
        return self.model is not None
    
    def generate_schedule(self, prompt):
        if not self.is_ready():
            return None
        try:
            response = self.model.generate_content(prompt)
            return self._extract_json(response.text)
        except:
            return None
    
    def _extract_json(self, text):
        try:
            return json.loads(text)
        except:
            patterns = [r'```json\s*(.*?)\s*```', r'\{.*\}']
            for p in patterns:
                match = re.search(p, text, re.DOTALL)
                if match:
                    try:
                        return json.loads(match.group(1) if 'json' in p else match.group())
                    except:
                        pass
        return None
'''

with open('ai_engine.py', 'w') as f:
    f.write(ai_engine_content)

print("✅ ai_engine.py created")

In [ ]:
# Create the main Gradio app
app_content = '''
"""NeuroFlow Gradio App"""

import gradio as gr
import json
from config import APP_NAME, APP_TAGLINE, ENERGY_PATTERNS, BREAK_STRUCTURES
from ai_engine import GeminiAIEngine

# CSS for styling
CSS = """
.header { text-align: center; padding: 2rem; background: linear-gradient(135deg, #6366F1, #8B5CF6); color: white; border-radius: 16px; margin-bottom: 2rem; }
.header h1 { font-size: 2.5rem; margin-bottom: 0.5rem; }
.metric-card { background: #f3f4f6; padding: 1.5rem; border-radius: 12px; text-align: center; }
.metric-value { font-size: 2rem; font-weight: bold; color: #6366F1; }
"""

def generate_prompt(subjects, exam_dates, difficulties, daily_hours, energy, stress, sleep, break_type):
    return f\"\"\"
Create a detailed study schedule for these subjects: {subjects}

EXAM DATES: {exam_dates}
DIFFICULTY LEVELS: {difficulties}
DAILY HOURS: {daily_hours}
ENERGY PATTERN: {energy}
STRESS LEVEL: {stress}/10
SLEEP HOURS: {sleep}
BREAK STRUCTURE: {break_type}

Create a JSON response with:
1. daily_schedule: array of time slots with time, subject, activity, duration_minutes, intensity
2. deep_work_blocks: focused study sessions
3. break_structure: work/break timing
4. subject_rotation: weekly plan
5. weekly_revision: review sessions
6. optimization_notes: suggestions

Also include cognitive_load_score (0-100), burnout_risk_percentage (0-100), efficiency_prediction (0-100), and recommendations.
\"\"\"

def format_schedule(data):
    if not data or 'daily_schedule' not in data:
        return "Error generating schedule"
    
    html = "<div style='background: white; border-radius: 12px; overflow: hidden;'>\"
    html += "<div style='background: linear-gradient(135deg, #6366F1, #8B5CF6); color: white; padding: 1rem;'><h3>📅 Your Study Schedule</h3></div>\"
    
    for slot in data['daily_schedule']:
        time_str = slot.get('time', '')
        subject = slot.get('subject', '')
        activity = slot.get('activity', '')
        duration = slot.get('duration_minutes', 0)
        
        if 'break' in activity.lower():
            bg = '#f3f4f6'
            border = '#9ca3af'
        else:
            bg = 'white'
            border = '#6366f1'
        
        html += f\"<div style='display: grid; grid-template-columns: 100px 1fr 80px; padding: 12px; border-bottom: 1px solid #e5e7eb; background: {bg}; border-left: 4px solid {border};'>\"
        html += f\"<div style='font-weight: 600; color: #6366f1;'>{time_str}</div>\"
        html += f\"<div><strong>{subject}</strong> <span style='color: #6b7280;'>({activity})</span></div>\"
        html += f\"<div style='text-align: right; color: #6b7280;'>{duration}m</div>\"
        html += "</div>\"
    
    html += "</div>\"
    return html

def format_metrics(data):
    if not data:
        return ""
    
    cognitive = data.get('cognitive_load_score', 50)
    burnout = data.get('burnout_risk_percentage', 50)
    efficiency = data.get('efficiency_prediction', 70)
    
    prod_score = max(0, min(100, 100 - (cognitive + burnout) / 2 + efficiency / 4))
    
    html = f\"""
    <div style='display: grid; grid-template-columns: repeat(4, 1fr); gap: 16px; margin: 20px 0;'>
        <div class='metric-card'><div class='metric-value'>{prod_score:.0f}</div><div>Productivity</div></div>
        <div class='metric-card'><div class='metric-value' style='color: {'#10b981' if burnout < 40 else '#ef4444'};'>{burnout:.0f}%</div><div>Burnout Risk</div></div>
        <div class='metric-card'><div class='metric-value'>{cognitive:.0f}</div><div>Cognitive Load</div></div>
        <div class='metric-card'><div class='metric-value' style='color: #10b981;'>{efficiency:.0f}%</div><div>Efficiency</div></div>
    </div>
    """\"
    return html

def format_analysis(data):
    if not data or 'recommendations' not in data:
        return ""
    
    html = "<div style='background: white; padding: 20px; border-radius: 12px;'><h3>🧠 Analysis & Recommendations</h3>\"
    
    for rec in data.get('recommendations', []):
        html += f\"<div style='padding: 12px; background: #f0fdf4; border-left: 4px solid #10b981; margin-bottom: 8px; border-radius: 0 8px 8px 0;'>{rec}</div>\"
    
    html += "</div>\"
    return html

def generate_schedule(subjects, exam_dates, difficulties, daily_hours, energy, stress, sleep, break_type):
    engine = GeminiAIEngine()
    
    if not engine.is_ready():
        return "⚠️ Please set your Google API Key", "", "", ""
    
    prompt = generate_prompt(subjects, exam_dates, difficulties, daily_hours, energy, stress, sleep, break_type)
    data = engine.generate_schedule(prompt)
    
    if not data:
        return "❌ Failed to generate schedule. Try again.", "", "", ""
    
    schedule_html = format_schedule(data)
    metrics_html = format_metrics(data)
    analysis_html = format_analysis(data)
    
    # Save to file for PDF export
    with open('schedule.json', 'w') as f:
        json.dump(data, f)
    
    return schedule_html, metrics_html, analysis_html, "✅ Schedule generated successfully!"

# Create Gradio interface
with gr.Blocks(css=CSS, title="NeuroFlow - AI Study Planner") as demo:
    gr.HTML(f\"""
    <div class='header'>
        <h1>{APP_NAME}</h1>
        <p>{APP_TAGLINE}</p>
        <p style='opacity: 0.8;'>Achieve 90%+ scores without burnout</p>
    </div>
    """\")
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## 📋 Your Information")
            
            subjects = gr.Textbox(label="Subjects", placeholder="Math, Physics, Chemistry")
            exam_dates = gr.Textbox(label="Exam Dates", placeholder="2024-06-15, 2024-06-20, 2024-06-25")
            difficulties = gr.Textbox(label="Difficulty (1-5)", placeholder="5, 4, 3")
            daily_hours = gr.Slider(1, 16, value=8, label="Daily Study Hours")
            energy = gr.Dropdown(list(ENERGY_PATTERNS.keys()), value="morning_high", label="Energy Pattern")
            break_type = gr.Dropdown(list(BREAK_STRUCTURES.keys()), value="Deep Focus (50-10)", label="Break Structure")
            stress = gr.Slider(1, 10, value=5, label="Stress Level")
            sleep = gr.Slider(3, 12, value=7, label="Sleep Hours")
            
            generate_btn = gr.Button("🚀 Generate My Study Plan", variant="primary")
        
        with gr.Column(scale=2):
            metrics_output = gr.HTML()
            
            with gr.Tabs():
                with gr.TabItem("📅 Schedule"):
                    schedule_output = gr.HTML()
                with gr.TabItem("🧠 Analysis"):
                    analysis_output = gr.HTML()
            
            status_output = gr.Textbox(label="Status", interactive=False)
    
    generate_btn.click(
        fn=generate_schedule,
        inputs=[subjects, exam_dates, difficulties, daily_hours, energy, stress, sleep, break_type],
        outputs=[schedule_output, metrics_output, analysis_output, status_output]
    )

if __name__ == "__main__":
    demo.launch(share=True, debug=True)
'''

with open('app.py', 'w') as f:
    f.write(app_content)

print("✅ app.py created")

## 🚀 Step 4: Launch NeuroFlow!

In [ ]:
# Launch the application
!python app.py

---

## 📖 How to Use

1. **Fill in your subjects** - Enter all subjects you're studying
2. **Add exam dates** - Format: YYYY-MM-DD
3. **Rate difficulty** - 1 (easy) to 5 (very hard)
4. **Set your preferences** - Energy pattern, stress level, sleep
5. **Click Generate** - AI creates your personalized plan

## 📊 Understanding Results

- **Productivity Score:** Overall effectiveness (higher is better)
- **Burnout Risk:** Chance of exhaustion (lower is better)
- **Cognitive Load:** Mental effort required (moderate is ideal)
- **Efficiency:** Predicted performance (higher is better)

## 💡 Tips

- Be honest about stress/sleep for accurate results
- Use realistic daily hours (don't overestimate)
- Regenerate as exam dates approach
- Follow the schedule consistently

---

**Made with ❤️ by NeuroFlow AI**